[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/361025-commodity-curve-dynamics.ipynb)

# Exploiting the dynamics of commodity futures curves — reproduction

**Robert J. Bianchi, John Hua Fan, Joëlle Miffre & Tingxi Zhang** · [arXiv 2308.00383](https://arxiv.org/abs/2308.00383) (2023)

The paper fits a **Nelson-Siegel** curve to each commodity's futures term structure every day — giving a *level*, *slope* and *curvature* — and bets that whatever the curve shape did yesterday, it does again today. Headline (§4.2, 1992–2019, 21 commodities): the **slope** book earns Sharpe **1.41**, the **curvature** book **1.27**, and the level book is unprofitable — with profits said to survive transaction costs and to be unrelated to known commodity risk factors.

This notebook runs the **same data and logic** as our engine and lands on the report's exact numbers. It loads a four-point curve panel built from exchange daily settlements (CME Globex, via Databento) — the nearest **four calendar contracts** for **16 of the 21** commodities, daily **2010-06→2026-06** — rebuilds the roll-clean leg returns, fits Nelson-Siegel daily, and trades the paper's own spreads.

**What you'll see:** the slope book measures **−0.56**, not +1.41; curvature **−4.29**, not +1.27; only the *negative* claim replicates (the level book is indeed unprofitable). Then the decisive diagnostic — the daily change in a calendar spread is dominated by each leg's **settlement noise**: metals sit at lag-1 autocorrelation −0.42 to −0.55, against the **−0.50 that differencing pure white noise produces by construction**, while the same markets' front-contract returns are clean. The effect dies at lag 2. And it doesn't matter which way you read the paper's slope-sign convention: the **flipped** book is positive gross but dies at 0.5–2bps against 1.37 turnover a day.

**Scope note:** the paper's own 1992–2019 sample has no free data and is untestable — and the paper itself concedes the edge was concentrated before the 2000 CFMA and weakened after, so our window *is* that weaker era. The full C0–C7 gate suite (deflation, cost, crisis, spanning) lives in the [report](https://market-graphs.com/praxis/papers/361025).

In [ ]:
# Pinned to the engine's versions so the numbers reproduce exactly.
%pip install -q numpy==2.1.2 pandas==2.2.3 pyarrow

In [ ]:
"""Praxis reproduction — paper 361025
"Exploiting the dynamics of commodity futures curves" (Bianchi, Fan, Miffre & Zhang, 2023)
arXiv 2308.00383

Self-contained: downloads the published four-point curve panel and reproduces the
Praxis report's exact numbers. Same data, same logic -> the report's figures.
"""
import numpy as np, pandas as pd

OOS_START, PPY, LAMBDA = "2010-08-01", 252, 9.0
CLAIMED = {"S": 1.41, "C": 1.27, "L": None}   # paper §4.2: L "unprofitable"

# ---------------------------------------------------------------- data --------
def load_panel():
    """The four-point curve panel: for each commodity and day, the nearest FOUR
    calendar contracts (close, instrument id, time to maturity). The panel derives
    from licensed Databento (CME GLBX + ICE) data and is NOT redistributed with
    this repository — rebuild it under your own Databento entitlement (the same
    adapter pattern Praxis ships to clients), or contact MarketGraphs, then place
    the file next to this notebook."""
    try:
        df = pd.read_parquet("361025-commodity-curve-panel.parquet")
    except FileNotFoundError:
        raise SystemExit(
            "361025-commodity-curve-panel.parquet not found — built from licensed Databento "
            "data and not redistributed; rebuild it under your own Databento entitlement, "
            "or contact MarketGraphs.")
    df["date"] = pd.to_datetime(df["date"])
    return df


def roll_clean_returns(price, iid):
    """The calendar ladder shifts one rung per roll, so on a roll day the contract
    now at depth-k was at depth-(k+1) yesterday:
        non-roll u:  price[k][u] / price[k][u-1]   - 1
        roll     u:  price[k][u] / price[k+1][u-1] - 1   (the aged contract)
    The deepest depth on a roll needs an unavailable 5th contract -> NaN.

    NOTE: computed PER COMMODITY on its own trading-day index, so `shift(1)` is
    that market's previous session — not the previous row of a unioned calendar."""
    out = {}
    for k in sorted(price):
        p = price[k]
        roll = iid[k].ne(iid[k].shift(1)) & iid[k].shift(1).notna()
        same = p / p.shift(1) - 1.0
        if k + 1 in price:
            out[k] = same.where(~roll, p / price[k + 1].shift(1) - 1.0)
        else:
            out[k] = same.where(~roll, np.nan)
    return out


def per_commodity(df, nm):
    """All four depths for one commodity, aligned on ITS OWN depth-0 date index."""
    sub = df[df.commodity == nm]
    price, iid, tau = {}, {}, {}
    for d in sorted(sub.depth.unique()):
        s = sub[sub.depth == d].drop_duplicates("date").set_index("date").sort_index()
        price[d], iid[d], tau[d] = s["close"], s["instrument_id"], s["tau_years"]
    idx = price[0].index
    for d in list(price):
        price[d] = price[d].reindex(idx)
        iid[d] = iid[d].reindex(idx)
        tau[d] = tau[d].reindex(idx)
    return price, iid, tau, idx


# ------------------------------------------------------- Nelson-Siegel --------
def ns_loadings(tau, lam):
    x = lam * tau
    with np.errstate(divide="ignore", invalid="ignore"):
        e = np.exp(-x)
        L1 = np.where(x > 1e-8, (1 - e) / x, 1.0)
        L2 = np.where(x > 1e-8, (1 - e) / x - e, 0.0)
    return np.stack([np.ones_like(L1), L1, L2], axis=-1)


def fit_betas(logp, tau, lam=LAMBDA):
    """(T,K) log-prices + maturities -> (T,3) level/slope/curvature at fixed decay.

    The paper specifies a decay "estimated daily". On a four-point curve that is
    not identifiable — the fit hops between values and all three factors absorb
    the wobble (they end up with identical autocorrelation). We fix it, the
    standard treatment for sparse curves. The free-decay variant fails too.
    """
    T = logp.shape[0]
    out = np.full((T, 3), np.nan)
    ok = np.isfinite(logp).all(1) & np.isfinite(tau).all(1)
    if ok.any():                                            # fast path: all 4 points
        X = ns_loadings(tau[ok], lam)                       # (G,K,3)
        out[ok] = np.einsum("gik,gk->gi", np.linalg.pinv(X), logp[ok])
    # thin markets don't quote the 4th contract every day (oats 50% of days,
    # lumber 66%, gold 81%) — fit those on whatever points exist, needing >=3
    for t in np.where(~ok)[0]:
        m = np.isfinite(logp[t]) & np.isfinite(tau[t])
        if m.sum() >= 3:
            X = ns_loadings(tau[t][m], lam)
            out[t] = np.linalg.pinv(X) @ logp[t][m]
    return out


# ------------------------------------------------------------ strategy --------
def sign_weights(sig):
    """Dollar-neutral equal weights: +1/(2 N_long), -1/(2 N_short); flat if a side is empty."""
    s = np.sign(sig.values)
    pos, neg = s > 0, s < 0
    nl = pos.sum(1, keepdims=True).astype(float)
    ns_ = neg.sum(1, keepdims=True).astype(float)
    w = np.zeros_like(s, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        np.divide(pos, 2 * nl, out=w, where=pos)
        np.divide(neg, -2 * ns_, out=w, where=neg)
    w[((nl < 1) | (ns_ < 1)).ravel()] = 0.0
    return pd.DataFrame(w, index=sig.index, columns=sig.columns)


def sharpe(r):
    r = r.loc[OOS_START:].dropna()
    return float(r.mean() / r.std(ddof=1) * np.sqrt(PPY))


def tstat(r):
    r = r.loc[OOS_START:].dropna()
    return float(r.mean() / r.std(ddof=1) * np.sqrt(len(r)))


def run():
    df = load_panel()
    names = sorted(df.commodity.unique())

    betas, rets = {}, {d: {} for d in range(4)}
    for nm in names:
        price, iid, tau, idx = per_commodity(df, nm)
        r = roll_clean_returns(price, iid)                 # on this market's own index
        for d in r:
            rets[d][nm] = r[d]
        Y = np.column_stack([np.log(price[d].values) for d in range(4)])
        T = np.column_stack([tau[d].values for d in range(4)])
        betas[nm] = pd.DataFrame(fit_betas(Y, T), index=idx,
                                 columns=["level", "slope", "curvature"])

    ret = {d: pd.DataFrame(rets[d]).sort_index() for d in range(4)}   # union afterwards

    # L = front outright · S = long front / short 4th · C = butterfly (-1,+2,0,-1)
    legs = {"L": ret[0], "S": ret[0] - ret[3], "C": -ret[0] + 2 * ret[1] - ret[3]}
    key = {"L": "level", "S": "slope", "C": "curvature"}

    out = {}
    for st in ("L", "S", "C"):
        beta = pd.DataFrame({nm: betas[nm][key[st]] for nm in names}).sort_index()
        w = sign_weights(beta.diff())                        # signal at close(t)
        lr = legs[st].reindex(index=w.index, columns=w.columns)
        out[st] = (w.shift(1) * lr).sum(axis=1, min_count=1).dropna()   # realised t->t+1

    print(f"{'book':26}{'claimed':>10}{'measured':>11}{'t':>8}")
    for st, label in (("S", "S  slope (headline)"), ("C", "C  curvature"), ("L", "L  level")):
        c = CLAIMED[st]
        print(f"{label:26}{(c if c is not None else 'unprofitable'):>10}"
              f"{sharpe(out[st]):>11.3f}{tstat(out[st]):>8.2f}")

    # the decisive diagnostic: is the daily signal economics, or settlement noise?
    print("\nlag-1 autocorrelation — differencing pure white noise gives EXACTLY -0.50")
    print(f"{'commodity':16}{'front ret':>11}{'spread':>9}{'butterfly':>11}")
    for nm in ["Silver", "Copper", "Gold", "Light crude oil", "Live cattle"]:
        r0, r1, r3 = ret[0][nm], ret[1][nm], ret[3][nm]
        print(f"{nm:16}{r0.autocorr(1):>11.3f}{(r0-r3).autocorr(1):>9.3f}"
              f"{(-r0+2*r1-r3).autocorr(1):>11.3f}")

    print("\nskip-a-day (a one-day noise bounce dies at lag 2; economics would persist)")
    for st in ("S", "C"):
        beta = pd.DataFrame({nm: betas[nm][key[st]] for nm in names}).sort_index()
        w = sign_weights(beta.diff())
        lr = legs[st].reindex(index=w.index, columns=w.columns)
        row = {f"lag{L}": round(sharpe((w.shift(L) * lr).sum(axis=1, min_count=1).dropna()), 3)
               for L in (1, 2, 3)}
        print(f"  {st}: {row}")

    # sign-robustness: even the flipped (noise-harvesting) book dies on costs
    print("\nflipped book (harvest the bounce) net of costs")
    for bps in (0.0, 1.0, 2.0):
        line = []
        for st in ("S", "C"):
            beta = pd.DataFrame({nm: betas[nm][key[st]] for nm in names}).sort_index()
            w = -sign_weights(beta.diff())
            lr = legs[st].reindex(index=w.index, columns=w.columns)
            gross = (w.shift(1) * lr).sum(axis=1, min_count=1)
            tc = w.diff().abs().sum(axis=1) * (bps * 1e-4) * 2.0
            line.append(f"{st} {sharpe((gross - tc.reindex(gross.index)).dropna()):+.2f}")
        print(f"  {bps:>4g}bps: " + "  ".join(line))
    return out



In [ ]:
import matplotlib.pyplot as plt

books = run()
fig, ax = plt.subplots(figsize=(11, 4.5))
for k, lab, c in (("S", "Slope book (headline)", "#A93226"),
                  ("C", "Curvature book", "#051C5B"),
                  ("L", "Level book", "#999999")):
    eq = (1 + books[k].loc[OOS_START:]).cumprod()
    ax.plot(eq.index, eq.values, label=lab, color=c, lw=1.4)
ax.axhline(1.0, color="k", lw=0.6, ls="--")
ax.set_yscale("log"); ax.set_ylabel("growth of 1 (log)")
ax.set_title("Paper 361025 — Nelson-Siegel curve-continuation books, 2010–2026")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()